# Entscheidsuche.ch API - Example Notebook

Explore search results, retrieve document metadata and files, and summarize a small sample with the `entscheidsuche` Python package.

This package is independently developed and is not official, endorsed by, associated with, or affiliated with entscheidsuche.ch.

Run cells in order: later examples reuse earlier results. Network examples contact entscheidsuche.ch, and section 4.3 writes a JSON file locally. Outputs are intentionally cleared; counts, results, and file availability depend on the service. See [the README](../README.md) for configuration and [API notes](../API-DOCUMENTATION.md) for parser behavior and validation limits.


## 1. Setup and Installation

From the repository root, run `uv sync`, then launch Jupyter with `uv run jupyter lab` and open this notebook. Select the repository's `.venv` kernel. The development dependencies include Jupyter and pandas.

The client reads `config.yaml` from the kernel's working directory when present; otherwise it uses built-in defaults. Pass `config_path` explicitly if you need the repository configuration from another directory. Constructor arguments override the corresponding settings.


In [ ]:
# Run `uv sync` from the repo root and select the repo `.venv` as this notebook's kernel.
import entscheidsuche

print(f"entscheidsuche {entscheidsuche.__version__} ready")

In [ ]:
# Import the package
from collections import defaultdict

import httpx

from entscheidsuche import EntscheidsucheClient, utils

# Create the client
client = EntscheidsucheClient()
print("Client created successfully!")

## 2. Built-in Reference Lists

### 2.1 List Canton and Federal Codes

`list_cantons()` returns the package's built-in mapping of the 26 cantons plus `CH` for the federal level. This lookup makes no network request and does not measure coverage in the search index.


In [ ]:
# List all cantons
cantons = client.list_cantons()
print("Available cantons:")
for code, name in sorted(cantons.items()):
    print(f"  {code}: {name}")

### 2.2 List Known Scrapers

`list_scrapers()` returns a static list bundled with the package, not a live inventory or status check. A scraper (also called a spider) identifies a source used to collect decisions. The example groups names by their prefix, including federal (`CH`) and upload (`XX`) entries.


In [ ]:
# List all known scrapers
scrapers = client.list_scrapers()
print(f"Number of scrapers: {len(scrapers)}")
print("\nScrapers by canton:")

# Group by canton
by_canton = defaultdict(list)
for s in scrapers:
    canton = s.split("_")[0]
    by_canton[canton].append(s)

for canton in sorted(by_canton.keys()):
    print(f"  {canton}: {', '.join(by_canton[canton])}")

## 3. Search API

### 3.1 Simple Text Search

`search()` returns a `SearchResult` with a page of hits and response metadata. The client sends the text as an Elasticsearch `query_string` with `AND` as the default operator. Each hit exposes a document ID through `signatur`, plus fields such as date, language, and multilingual title.

`size` controls the number of hits requested, not the total matches. `total` retains the service's reported count but not its exact/lower-bound indicator; use `search_raw()` when that distinction matters. `took_ms` is the server-reported search time, not the full request duration.


In [ ]:
# Simple search
results = client.search("Mietvertrag", size=5)

print(f"Found {results.total} results (showing first {len(results.hits)})")
print(f"Search took {results.took_ms}ms\n")

for hit in results.hits:
    print(f"📄 {hit.signatur}")
    score = f"{hit.score:.2f}" if hit.score is not None else "N/A"
    print(f"   Date: {hit.date} | Language: {hit.language} | Score: {score}")
    title = hit.title.get("de", "N/A")
    print(f"   Title: {title[:70]}..." if len(title) > 70 else f"   Title: {title}")
    print()

### 3.2 Search with Filters

Combine text with canton, language, and inclusive date bounds (`YYYY-MM-DD`). This example intentionally uses the fixed historical period 2020–2024; change the dates to explore another period. A `spider` filter can narrow the search to a specific source.


In [ ]:
# Search with filters
results = client.search(
    query="Arbeitsvertrag Kündigung",
    size=10,
    canton="ZH",  # Only Zürich
    language="de",  # Only German
    date_from="2020-01-01",  # From 2020
    date_to="2024-12-31",  # To 2024
)

print(f"Found {results.total} results from Zürich in German (2020-2024)\n")

for hit in results.hits:
    print(f"📄 {hit.signatur} ({hit.date})")

### 3.3 Search with Sorting

Sort matching decisions by their indexed `date`, newest first. The client passes the indexed field through without verifying its upstream meaning; the example requests only five hits.


In [ ]:
# Search sorted by date (newest first)
results = client.search(
    query="Erbrecht",
    size=5,
    sort_by="date",
    sort_order="desc",
)

print("Newest Erbrecht decisions:")
for hit in results.hits:
    print(f"  {hit.date}: {hit.signatur}")

### 3.4 Pagination

`from_` is a zero-based offset and `size` is the page size. The two calls below request successive five-hit pages. These are separate searches, so changes to the live index can affect page boundaries; this example is not a complete export workflow.


In [ ]:
# Pagination example
page_size = 5

# Page 1
page1 = client.search("Strafrecht", size=page_size, from_=0)
print(f"Page 1 (results 1-{len(page1.hits)} of {page1.total}):")
for hit in page1.hits:
    print(f"  - {hit.signatur}")

# Page 2
page2 = client.search("Strafrecht", size=page_size, from_=page_size)
print(f"\nPage 2 (results {page_size + 1}-{page_size + len(page2.hits)}):")
for hit in page2.hits:
    print(f"  - {hit.signatur}")

### 3.5 Raw Elasticsearch Query

`search_raw()` sends a query dictionary to the configured search endpoint and returns its JSON response without converting it to `SearchResult`. Supported fields and query features depend on the service.

This example searches German abstracts, requires an indexed language field, selects a subset of source fields, and requests highlighting. It prints the first hit's ID and source; any highlights remain in the raw response.


In [ ]:
# Advanced: raw Elasticsearch query
raw_query = {
    "size": 5,
    "query": {
        "bool": {
            "must": [{"match": {"abstract.de": "Datenschutz"}}],
            "filter": [{"exists": {"field": "attachment.language"}}],
        }
    },
    "_source": ["title", "date", "hierarchy", "abstract"],
    "highlight": {"fields": {"abstract.de": {}}},
}

raw_results = client.search_raw(raw_query)
print(f"Raw query returned {raw_results['hits']['total']} results")
if raw_results["hits"]["hits"]:
    first = raw_results["hits"]["hits"][0]
    print(f"First result ID: {first['_id']}")
    print(f"First result source: {first['_source']}")

## 4. Document Access

### 4.1 Get Document Metadata (JSON)

Search for one `CH_BGE` hit, then pass `hit.spider` and `hit.signatur` to the document methods. Here, `hit.signatur` is the full document ID. The returned metadata's `doc.signatur` may be a shorter series identifier; `doc.document_id` preserves the full ID needed for subsequent retrieval.

`get_document_json()` returns a `CaseDocument`, with multilingual text normalized into language-keyed dictionaries. HTML and PDF URLs can be absent. The next two examples reuse this search result.


In [ ]:
# First, get a document signatur from search
results = client.search("BGE", size=1, spider="CH_BGE")

if results.hits:
    hit = results.hits[0]
    print(f"Getting document: {hit.signatur} from {hit.spider}")

    # Get full JSON metadata
    doc = client.get_document_json(hit.spider, hit.signatur)

    print("\nDocument details:")
    print(f"  Signatur: {doc.signatur}")
    print(f"  Spider: {doc.spider}")
    print(f"  Date: {doc.date}")
    print(f"  Language: {doc.language}")
    print(f"  Case numbers: {doc.num}")
    print(f"  Original URL: {doc.url}")
    print(f"  HTML URL: {doc.html_url}")
    print(f"  PDF URL: {doc.pdf_url}")
    print(f"  Kopfzeile (DE): {doc.kopfzeile.get('de', 'N/A')[:100]}...")
else:
    print("No documents found")

### 4.2 Get Document HTML

`get_document_html()` first retrieves metadata to resolve the HTML path, then returns the content as a string. The example prints a short source preview. Its `HTTPError` handler reports both missing content and request failures. For PDF bytes, use `get_document_pdf()` with the same identifiers.


In [ ]:
# Get HTML content (if available)
if results.hits:
    hit = results.hits[0]
    try:
        html = client.get_document_html(hit.spider, hit.signatur)
        print(f"HTML content length: {len(html)} characters")
        print("\nFirst 500 characters:")
        print(html[:500])
    except httpx.HTTPError as e:
        print(f"Could not fetch HTML: {e}")

### 4.3 Download Documents

This example requests only JSON metadata and saves it under `./downloads`, relative to the kernel's working directory. Rerunning it for the same document overwrites that file. Add `"html"` or `"pdf"` to `formats` to request those files as well.

`download_document()` returns a mapping of formats actually saved to their paths. It skips HTTP 404 responses; other request errors propagate. Omitting `formats` uses the configured format list, whose built-in default is JSON, HTML, and PDF.


In [ ]:
from pathlib import Path

# Download documents to local directory
if results.hits:
    hit = results.hits[0]
    output_dir = Path("./downloads")

    saved = client.download_document(
        spider=hit.spider,
        signatur=hit.signatur,
        output_dir=output_dir,
        formats=["json"],  # Only JSON to avoid large downloads in example
    )

    print("Downloaded files:")
    for fmt, path in saved.items():
        print(f"  {fmt}: {path}")

## 5. Index and Jobs Files

### 5.1 Get Index File

`get_index("CH_BGE")` reads that scraper's latest index file. The client aggregates counts and change lists from its `signaturen` entries. These describe the returned scraper data, not the total number of decisions across the service.


In [ ]:
# Get latest index for a scraper
index = client.get_index("CH_BGE")

print("Index for CH_BGE:")
print(f"  Job type: {index.jobtyp}")
print(f"  Time: {index.time}")
print(f"  Total documents: {index.total_count}")
print(f"  New documents: {len(index.new_documents)}")
print(f"  Updated documents: {len(index.updated_documents)}")
print(f"  Deleted documents: {len(index.deleted_documents)}")

### 5.2 Get Jobs File

`get_jobs("CH_BGE")` reads the scraper's latest jobs file and exposes its `signaturen` mapping as `jobs.documents`. The printed count is the number of entries in that mapping; the example does not inspect individual statuses or retrieve a history of runs.


In [ ]:
# Get latest jobs file for a scraper
jobs = client.get_jobs("CH_BGE")

print("Jobs for CH_BGE:")
print(f"  Job type: {jobs.jobtyp}")
print(f"  Time: {jobs.time}")
print(f"  Number of signatures: {len(jobs.documents)}")

## 6. Blocklist

`get_blocklist()` returns document IDs from the service's blocklist, flattening a scraper-to-IDs mapping when supplied. Fetching this list does not automatically filter search results or prevent downloads; callers must apply any exclusion themselves.


In [ ]:
# Get blocklist of removed documents
try:
    blocklist = client.get_blocklist()
    print(f"Blocked documents: {len(blocklist)}")
    if blocklist:
        print(f"First 5: {blocklist[:5]}")
except httpx.HTTPError as e:
    print(f"Could not fetch blocklist: {e}")

## 7. Sitemap Index

`get_sitemap_index()` returns sitemap URLs from the site's sitemap index. This cell prints the first ten URLs; it does not fetch those sitemaps or enumerate their document links.


In [ ]:
# Get sitemap index
sitemaps = client.get_sitemap_index()

print(f"Number of sitemaps: {len(sitemaps)}")
print("\nFirst 10 sitemaps:")
for sm in sitemaps[:10]:
    print(f"  {sm}")

## 8. Utility Functions

### 8.1 Parse Case Numbers

`parse_case_number()` recognizes the two federal case-number patterns shown below. For other forms, including `VB.2023.00123`, it only attempts to extract a four-digit year and leaves the other components as `None`. It preserves the input in `original`; it is not a general validator for Swiss case numbers.


In [ ]:
# Parse Swiss case numbers
case_numbers = [
    "4A_123/2023",  # Federal Court
    "A-1234/2022",  # Federal Administrative Court
    "VB.2023.00123",  # Cantonal court
]

for cn in case_numbers:
    parsed = utils.parse_case_number(cn)
    print(f"\n{cn}:")
    for key, value in parsed.items():
        print(f"  {key}: {value}")

### 8.2 Build Search Queries

`build_search_query()` constructs a query dictionary locally without sending a request. This example combines text, canton, year, and language constraints. To execute it, pass the dictionary to `client.search_raw()`; add a `size` key if you want to specify the result-page size.


In [ ]:
import json

# Build a search query programmatically
query = utils.build_search_query(
    text="Vertragsrecht",
    canton="BE",
    year=2023,
    language="de",
)

print("Generated Elasticsearch query:")
print(json.dumps(query, indent=2))

### 8.3 Convert Search Results to a DataFrame

Fetch up to ten hits and convert that page to a pandas DataFrame. The helper does not retrieve additional pages or full document metadata. Pandas is included by `uv sync` for this repository's development environment, but is not a runtime dependency of the client package.


In [ ]:
# Convert to pandas DataFrame (requires pandas)
try:
    results = client.search("Haftung", size=10)
    df = utils.search_results_to_dataframe(results)

    print(f"DataFrame with {len(df)} rows:")
    if not df.empty:
        display(df[["signatur", "date", "language", "score"]].head())
except ImportError:
    print("pandas not installed. Run uv sync from the repository root and select its environment.")

### 8.4 Extract Text from HTML

This local example strips tags, script/style blocks, and excess whitespace, and decodes a fixed set of common HTML entities. It is a simple text-preview helper: document structure and formatting are lost, and it is not a full HTML parser.


In [ ]:
# Extract plain text from HTML
sample_html = """
<html>
<body>
<h1>Urteil vom 15. März 2023</h1>
<p>Der Beschwerdeführer macht geltend, dass...</p>
<p>Die Vorinstanz hat zu Recht festgestellt, dass...</p>
</body>
</html>
"""

text = utils.extract_text_from_html(sample_html)
print("Extracted text:")
print(text)

### 8.5 Group a Sample of Documents

Fetch up to 50 search hits and convert them to `CaseDocument` objects using the fields already present in each hit. `to_case_document()` makes no additional requests and does not fetch complete JSON metadata. The example groups this sample by canton/source prefix and year; it does not apply a local filter.


In [ ]:
# Get some documents
results = client.search("Recht", size=50)
documents = [hit.to_case_document() for hit in results.hits]

# Group by canton
by_canton = utils.group_by_canton(documents)
print("Documents by canton:")
for canton, docs in sorted(by_canton.items(), key=lambda x: -len(x[1]))[:5]:
    print(f"  {canton}: {len(docs)} documents")

# Group by year
by_year = utils.group_by_year(documents)
print("\nDocuments by year:")
for year, docs in sorted(by_year.items(), reverse=True)[:5]:
    print(f"  {year}: {len(docs)} documents")

### 8.6 Get Sample Statistics

`get_statistics()` summarizes the `documents` list from section 8.5, including language counts and the span of known years. These are statistics for the retrieved search page, not estimates of the whole collection. The canton grouping can include `CH` or other source prefixes as well as canton codes.


In [ ]:
# Get statistics for a set of documents
stats = utils.get_statistics(documents)

print(f"Statistics for {stats['total']} documents:")
print(f"  Cantons covered: {stats['cantons_count']}")
print(f"  Years span: {stats['years_span']}")
print(f"\n  By language: {stats['by_language']}")

## 9. Context Manager Usage

Use `with EntscheidsucheClient() as client:` to close that client's HTTP resources automatically, including when an exception occurs. This cell first calls `client.close()` to release the shared client created in section 1: the `with` statement creates a separate instance and reuses the variable name. After this example, that variable refers to a closed client; rerun setup before returning to earlier search cells.


In [ ]:
# Release the shared client before creating a separate instance.
client.close()

# Use as context manager for proper cleanup
with EntscheidsucheClient() as client:
    results = client.search("Bundesgericht", size=3)
    print(f"Found {results.total} results")

print("Client properly closed")

## 10. Rate Limiting

The synchronous client spaces request starts within each instance. The built-in delay is 0.5 seconds and can be changed in `config.yaml` or with `rate_limit_delay`. The example uses a one-second minimum interval and a 30-second HTTP timeout; a slow request can make the observed interval longer.

This delay is local to each client, not a shared limit across clients or processes. The cell creates a new client and closes it at the end.


In [ ]:
import time

# Create client with custom rate limiting
with EntscheidsucheClient(
    timeout=30.0,
    rate_limit_delay=1.0,  # 1 second between request starts
) as client:
    start = time.monotonic()
    for i in range(3):
        results = client.search("Test", size=1)
        print(f"Request {i + 1} completed at {time.monotonic() - start:.1f}s")

## Summary

The examples cover searching and paging through results, retrieving metadata and available document formats, inspecting scraper index/jobs data, reading blocklist and sitemap URLs, and summarizing a small result sample with utility functions.

Use modest page sizes and request rates, and close clients when finished. Credit entscheidsuche.ch as the data source where appropriate. The client's MIT license covers its code, not the returned decisions or other source content.
